In [1]:
import pandas as pd 
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)

In [3]:
df = pd.read_csv('fmnist_small.csv')

In [4]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# defining transformation
from torchvision.transforms import transforms

In [8]:
custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [9]:
from PIL import Image
import numpy as np

In [10]:
class CustomDataset(nn.Module):
    def __init__(self, features, labels, transform):
        self.features = features
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        # resize to (28, 28)
        image = self.features[index].reshape(28, 28)

        # convert to int8
        image = image.astype(np.uint8)

        # change black and white into color
        image = np.stack([image]*3, axis=-1)

        # convert array to PIL image
        image = Image.fromarray(image)

        # apply transformation
        image = self.transform(image)

        return image, torch.tensor(self.labels[index],dtype=torch.long)


In [11]:
train_dataset = CustomDataset(X_train, y_train, custom_transform)
test_dataset = CustomDataset(X_test, y_test, custom_transform)

In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [13]:
# fetch the pretrained model
import torchvision.models as models

vgg16 = models.vgg16(pretrained=True)

/home/roben/Codes/PracticalDeepLearning/practicaldeeplearning/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/roben/Codes/PracticalDeepLearning/practicaldeeplearning/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/roben/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:23<00:00, 23.7MB/s] 


In [19]:
vgg16.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=1024, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=1024, out_features=512, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=512, out_features=10, bias=True)
)

In [17]:
for param in vgg16.features.parameters():
    param.requires_grad=False

In [18]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10)
)

In [20]:
learning_rate = 0.001
epochs = 10

In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.classifier.parameters(), lr=learning_rate)

In [22]:
# Training loop
for epoch in range(epochs):
    total_epoch_loss = 0

    for batch_features, batch_label in train_loader:
        # forward
        outputs = vgg16(batch_features)

        # loss
        loss = criterion(outputs, batch_label)

        optimizer.zero_grad()

        # backward
        loss.backward()

        # update grad
        optimizer.step()
    
        total_epoch_loss = total_epoch_loss + loss.item()

    print(f"Epoch: {epoch+1}, Loss: {total_epoch_loss/(len(train_loader))}")

KeyboardInterrupt: 